# SenGuard-MSC K: production audit
Checks all twelve real trajectories before analysis. No MD gate result or website publication is produced here.


In [ ]:
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'mdtraj==1.11.2'], check=True)
source = '"""Audit the twelve production runs inside Kaggle, before any pooled analysis.\n\nReads actual XTC timestamps in bounded chunks, verifies topology atom counts,\nand saves an audit even on failure. Requires mdtraj. This is not the MD gate.\n"""\nimport argparse\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nSYSTEMS = (\'wt\', \'r24p\', \'d84n\', \'h123q\')\n\n\ndef check_times(times):\n    import numpy as np\n    t = np.asarray(times, dtype=float)\n    problems = []\n    if len(t) < 2 or not np.isfinite(t).all():\n        return [\'Trajectory requires at least two finite timestamps.\']\n    if abs(t[0]) > .1:\n        problems.append(\'Trajectory does not include production time zero; continuation history is missing.\')\n    differences = np.diff(t)\n    if (differences <= 0).any():\n        problems.append(\'Duplicate or reversed trajectory timestamps.\')\n    if not np.allclose(differences, 50, atol=.1, rtol=0):\n        problems.append(\'Trajectory does not have the planned uninterrupted 50 ps frame interval.\')\n    return problems\n\n\ndef topology_atom_count(path):\n    active, count = False, 0\n    for line in path.read_text().splitlines():\n        clean = line.split(\';\')[0].strip()\n        if clean.startswith(\'[\'):\n            active = clean.strip(\'[] \') == \'atoms\'\n        elif active and clean and not clean.startswith(\'#\'):\n            count += 1\n    return count\n\n\ndef audit(root, out):\n    from mdtraj.formats import XTCTrajectoryFile\n    import numpy as np\n    result = {\'status\': \'not_ready\', \'protocol\': \'ANALYSIS_AMENDMENT_4_MD.md\', \'runs\': {}, \'errors\': []}\n    for system in SYSTEMS:\n        for rep in range(1, 4):\n            rid = f\'{system}-{rep}\'\n            found = list(root.glob(f\'**/runs/{rid}/md.tpr\'))\n            row = {\'seed\': 2026 + 100 * SYSTEMS.index(system) + rep, \'errors\': []}\n            result[\'runs\'][rid] = row\n            if len(found) != 1:\n                row[\'errors\'].append(f\'Expected one run directory, found {len(found)}. Select a single version of each kernel.\')\n                continue\n            directory = found[0].parent\n            row[\'source\'] = str(directory)\n            row[\'tpr_sha256\'] = hashlib.sha256(found[0].read_bytes()).hexdigest()\n            if not (directory / \'md.cpt\').is_file():\n                row[\'errors\'].append(\'No restart checkpoint.\')\n            tops = [directory / f\'topol_Protein_chain_{c}.itp\' for c in \'AB\']\n            if not all(p.is_file() for p in tops):\n                row[\'errors\'].append(\'Missing chain topology; cannot verify trajectory atom identity.\')\n                continue\n            expected_atoms = sum(topology_atom_count(p) for p in tops)\n            trajectories = sorted(directory.glob(\'md*.xtc\'))\n            row[\'trajectory_files\'] = [p.name for p in trajectories]\n            times, counts = [], set()\n            for path in trajectories:\n                try:\n                    with XTCTrajectoryFile(str(path)) as reader:\n                        while True:\n                            xyz, t, step, box = reader.read(n_frames=10)\n                            if len(t) == 0:\n                                break\n                            counts.add(xyz.shape[1])\n                            times.extend(float(x) for x in t)\n                            if not np.isfinite(xyz).all():\n                                row[\'errors\'].append(f\'Nonfinite coordinates in {path.name}\')\n                                break\n                except Exception as e:\n                    row[\'errors\'].append(f\'{path.name}: {type(e).__name__}: {e}\')\n            row[\'frames\'] = len(times)\n            row[\'protein_atoms\'] = expected_atoms\n            row[\'observed_atom_counts\'] = sorted(counts)\n            if counts != {expected_atoms}:\n                row[\'errors\'].append(\'XTC atom count differs from the two protein topologies.\')\n            row[\'errors\'].extend(check_times(times))\n            row[\'duration_ns\'] = max(times) / 1000 if times else None\n            logtext = \'\\n\'.join(p.read_text(errors=\'replace\') for p in directory.glob(\'md*.log\'))\n            row[\'lincs_warnings\'] = len(re.findall(r\'LINCS WARNING\', logtext))\n            if row[\'lincs_warnings\'] or \'Fatal error\' in logtext:\n                row[\'errors\'].append(\'Production log contains LINCS warnings or a fatal error; inspect before analysis.\')\n            if not logtext:\n                row[\'errors\'].append(\'No production log available.\')\n    for rid, row in result[\'runs\'].items():\n        result[\'errors\'].extend(f\'{rid}: {message}\' for message in row[\'errors\'])\n    if not result[\'errors\']:\n        common = min(row[\'duration_ns\'] for row in result[\'runs\'].values())\n        result.update(status=\'ready_for_analysis\', common_duration_ns=common,\n                      analysis_window_ns=[common / 2, common], gate_minimum_met=common >= 15,\n                      target_25ns_met=common >= 25)\n        if common < 15:\n            result[\'note\'] = \'All runs are readable, but the pre-registered gate is untested below 15 ns. Resume before the final analysis if quota allows.\'\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(result, indent=2, allow_nan=False) + \'\\n\')\n    print(json.dumps(result, indent=2))\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\', type=Path, default=Path(\'/kaggle/input\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'/kaggle/working/md_audit.json\'))\n    args = parser.parse_args()\n    report = audit(args.input, args.output)\n    raise SystemExit(0 if report[\'status\'] == \'ready_for_analysis\' else 1)\n'
Path('/kaggle/working/audit_md_runs.py').write_text(source)
subprocess.run([sys.executable, '/kaggle/working/audit_md_runs.py'], check=True)
